# Pipeline CV Data Cleaning & Standarisasi

Notebook ini menjalankan proses cleaning end-to-end dari raw PDF / Excel CV
menjadi dataset siap pakai dengan empat kolom standar:
`pendidikan`, `pengalaman`, `skill`, dan `detail`.

| # | Tahap | Deskripsi | Checkpoint |
|---|-------|-----------|------------|
| 1 | OCR & Segmentasi CV PDF | Ekstrak teks PDF → seksi Pendidikan / Pengalaman / Skills / Lainnya | `hasil_cv_kaggle_it_ocr.xlsx` |
| 2A | Standarisasi Pendidikan | Normalisasi level pendidikan dari teks bebas | *(in-memory)* |
| 2B | Standarisasi Skill | Ekstraksi & standarisasi skill via kamus IT | *(in-memory)* |
| 2C | Standarisasi Pengalaman | Normalisasi lama pengalaman ke kategori | *(in-memory)* |
| 2D | Enrichment via Groq API | Ekstrak kolom `detail` dari teks Lainnya | `detail_checkpoint_dataset_cv_ocr.xlsx` |
| 3 | Dataset Tambahan (Kaggle) | Pipeline serupa untuk dataset resume teks | `detail_checkpoint_dataset_tambahan_cv.xlsx` |
| 4 | Gabung & Bersihkan Kolom `detail` | Merge, regex cleaning, normalisasi | *(in-memory)* |
| 5 | Deduplikasi & Simpan Final | Dedup berdasarkan skill, simpan output | `dataset_cv.xlsx` |

---
**Cara lanjut tanpa run ulang dari awal:**
Setiap cell batch processing membaca checkpoint secara otomatis.
Ganti nama `CHECKPOINT` ke file lama untuk melanjutkan, atau nama baru untuk mulai ulang.

---
# Import Library

Jalankan cell ini **satu kali** sebelum menjalankan bagian mana pun.

In [1]:
# !pip install pdfplumber pdf2image pytesseract groq openpyxl

import os
import re
import time
import getpass

import numpy as np
import pandas as pd
import pdfplumber
import pytesseract
from pdf2image import convert_from_path
from PIL import Image
from groq import Groq
from collections import Counter

print('✅ Semua library berhasil diimport')

✅ Semua library berhasil diimport


---
# BAGIAN 1 — OCR & Segmentasi CV PDF

Bagian ini membaca semua file PDF di folder yang ditentukan,
mengekstrak teks (otomatis memilih pdfplumber atau OCR),
lalu memisahkan teks ke empat seksi: **Pendidikan**, **Pengalaman**, **Skills**, **Lainnya**.

> **Kalau OCR sudah pernah dijalankan sebelumnya**, lewati seluruh Bagian 1
> dan langsung load file `hasil_cv_kaggle_it_ocr.xlsx` di cell *[Opsi lanjut]* (1.5).

Alur proses:
1. Konfigurasi path & demo limit
2. Definisi fungsi ekstraksi teks
3. Definisi fungsi parser seksi CV
4. Eksekusi loop atas semua PDF

## 1.1 Konfigurasi Path & Parameter

Masukkan path Tesseract, Poppler, folder PDF, dan nama file output.
`DEMO_LIMIT` membatasi jumlah file yang diproses. Ubah ke `None` untuk proses semua data.

In [ ]:
pytesseract.pytesseract.tesseract_cmd = r'C:\Program Files\Tesseract-OCR\tesseract.exe'
POPPLER_PATH = r'C:\Program Files\poppler\Library\bin'
OCR_LANG     = 'ind+eng'

FOLDER_CV       = r'C:\cv_kaggle_it_pdf'
OCR_OUTPUT_FILE = 'hasil_cv_kaggle_it2.xlsx'

# Ubah ke None untuk proses semua file
DEMO_LIMIT = 5

print(f'✅ Konfigurasi OK. Demo limit: {DEMO_LIMIT} file.')

✅ Konfigurasi OK. Demo limit: 5 file.


## 1.2 Fungsi Ekstraksi Teks PDF

Dua strategi ekstraksi:
- **pdfplumber** : untuk PDF teks-based (lebih cepat, lebih akurat)
- **OCR via pytesseract** : untuk PDF image-based (hasil scan/foto)

Fungsi `is_pdf_image_based()` mendeteksi secara otomatis strategi mana yang diperlukan
dengan menghitung karakter pada beberapa halaman pertama.

In [ ]:
def is_pdf_image_based(pdf_path, sample_pages=2):
    """Deteksi apakah PDF image-based (hasil scan/konversi gambar)"""
    total_chars = 0
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages[:sample_pages]:
                text = page.extract_text(x_tolerance=2, y_tolerance=2)
                if text:
                    total_chars += len(text.strip())
    except Exception:
        pass
    return total_chars < (30 * sample_pages)


def extract_text_via_ocr(pdf_path):
    """Konversi PDF ke gambar lalu jalankan OCR (untuk image-based PDF)"""
    full_text = ''
    try:
        pages = convert_from_path(pdf_path, dpi=300, poppler_path=POPPLER_PATH)
        for page_image in pages:
            gray = page_image.convert('L')
            text = pytesseract.image_to_string(gray, lang=OCR_LANG)
            if text:
                full_text += text + '\n'
    except Exception as e:
        print(f'[-] Error OCR {os.path.basename(pdf_path)}: {e}')
    return full_text


def extract_text_from_pdf(pdf_path):
    """Ekstrak teks dari PDF; otomatis pilih pdfplumber atau OCR"""
    full_text = ''
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                text = page.extract_text(x_tolerance=2, y_tolerance=2)
                if text:
                    full_text += text + '\n'
    except Exception as e:
        print(f'[-] Error pdfplumber {os.path.basename(pdf_path)}: {e}')

    if is_pdf_image_based(pdf_path):
        print('      [OCR] Terdeteksi image-based PDF, menjalankan OCR...', end=' ')
        full_text = extract_text_via_ocr(pdf_path)
    return full_text


print('✅ Fungsi ekstraksi teks siap.')

✅ Fungsi ekstraksi teks siap.


## 1.3 Fungsi Parser Seksi CV

`parse_cv_sections()` memisahkan teks CV ke empat seksi berdasarkan pencocokan
header baris menggunakan regex.

Mapping header yang dikenali:
- **Pengalaman**: `experience`, `work experience`, `pengalaman kerja`, dll.
- **Pendidikan**: `education`, `academic background`, `pendidikan`, dll.
- **Skills**: `skills`, `core competencies`, `keahlian`, `technical skills`, dll.
- **Lainnya**: semua baris yang tidak masuk seksi di atas

In [4]:
def parse_cv_sections(text):
    """Pisahkan teks CV ke seksi: Pendidikan, Pengalaman, Skills, Lainnya."""
    cv_data = {'Pendidikan': '', 'Pengalaman': '', 'Skills': '', 'Lainnya': ''}
    header_mapping = {
        r'^(experience|work experience|pengalaman kerja|pengalaman|employment history|work history|professional experience|relevant experience)': 'Pengalaman',
        r'^(education|academic background|pendidikan|latar belakang pendidikan|riwayat pendidikan|education and training|education & training)': 'Pendidikan',
        r'^(skills|core competencies|keahlian|highlights|technical skills|kemampuan|kompetensi|skills & competencies|key skills|hard skills|expertise|professional skills)': 'Skills',
    }
    current_section = 'Lainnya'
    for line in text.split('\n'):
        clean_line = line.strip().lower()
        if not clean_line:
            continue
        is_header = False
        for pattern, section_key in header_mapping.items():
            if re.match(pattern, clean_line):
                current_section = section_key
                is_header = True
                break
        if not is_header:
            cv_data[current_section] += line.strip() + '\n'
    return {k: v.strip() for k, v in cv_data.items()}


print('✅ Fungsi parser seksi CV siap.')

✅ Fungsi parser seksi CV siap.


## 1.4 Eksekusi: Loop atas Semua PDF

Iterasi semua file PDF di `FOLDER_CV`, ekstrak teks,
parse ke seksi, dan simpan hasilnya ke `OCR_OUTPUT_FILE`.

In [5]:
if not os.path.exists(FOLDER_CV):
    raise FileNotFoundError(f'Folder tidak ditemukan: {FOLDER_CV}')

pdf_files = sorted([f for f in os.listdir(FOLDER_CV) if f.lower().endswith('.pdf')])
if DEMO_LIMIT:
    pdf_files = pdf_files[:DEMO_LIMIT]
    print(f'DEMO MODE: memproses {len(pdf_files)} file.')
else:
    print(f'Ditemukan {len(pdf_files)} file PDF.')

all_cv_data = []
print(f"\n{'No.':<5} {'Nama File':<40} {'Status'}")
print('-' * 65)

for idx, filename in enumerate(pdf_files, 1):
    file_path = os.path.join(FOLDER_CV, filename)
    print(f'{idx:<5} {filename:<40}', end=' ')
    raw_text = extract_text_from_pdf(file_path)
    if not raw_text.strip():
        print('Gagal (teks kosong)')
        continue
    parsed = parse_cv_sections(raw_text)
    parsed['Nama File'] = filename
    all_cv_data.append(parsed)
    print('Selesai')

print('-' * 65)

df_ocr = pd.DataFrame(all_cv_data)[['Nama File', 'Pendidikan', 'Pengalaman', 'Skills', 'Lainnya']]
df_ocr.to_excel(OCR_OUTPUT_FILE, index=False)
print(f'\n✅ Hasil OCR disimpan: {OCR_OUTPUT_FILE}  |  Shape: {df_ocr.shape}')
df_ocr.head(3)

DEMO MODE: memproses 5 file.

No.   Nama File                                Status
-----------------------------------------------------------------
1     0(1).pdf                                       [OCR] Terdeteksi image-based PDF, menjalankan OCR... Selesai
2     0(2).pdf                                       [OCR] Terdeteksi image-based PDF, menjalankan OCR... Selesai
3     0(3).pdf                                       [OCR] Terdeteksi image-based PDF, menjalankan OCR... Selesai
4     0(4).pdf                                       [OCR] Terdeteksi image-based PDF, menjalankan OCR... Selesai
5     0(5).pdf                                       [OCR] Terdeteksi image-based PDF, menjalankan OCR... Selesai
-----------------------------------------------------------------

✅ Hasil OCR disimpan: hasil_cv_kaggle_it2.xlsx  |  Shape: (5, 5)


,Nama File,Pendidikan,Pengalaman,Skills,Lainnya
0,0(1).pdf,TRAINING\nCERTIFICATIONS\nJessica Claire\n@ 10...,shell scripting. Extensive experience in softw...,,SUMMARY
1,0(2).pdf,TRAINING\nFlorence Darlington Technical\nSchoo...,"Year Up - Information Technology\nLincoln, RI ...",+ Enterprise platforms\n+ Knowledge of Product...,JESSICA\nCLAIRE\n@ resumesample@example.com\n&...
2,0(3).pdf,"Caci International Inc. | Fort Detrick, MD\nEx...",,"Ab-initio GDE 3.1.5, Co >Opsys (3.0 f02.15), T...",Jessica Claire\n@ 100 Montgomery St. 10th Floo...


## 1.5 [Opsi Lanjut] Load Hasil OCR yang Sudah Ada

Cell ini dijalankan untuk **melanjutkan dari hasil OCR sebelumnya**
tanpa menjalankan ulang proses ekstraksi PDF di atas.

In [6]:
df_ocr = pd.read_excel('../dataset_mentah/hasil_cv_kaggle_it_ocr.xlsx')
print(f'✅ Dimuat dari file: hasil_cv_kaggle_it_ocr.xlsx  |  Shape: {df_ocr.shape}')
df_ocr.head(3)

✅ Dimuat dari file: hasil_cv_kaggle_it_ocr.xlsx  |  Shape: (3402, 5)


,Nama File,Pendidikan,Pengalaman,Skills,Lainnya
0,0(1).pdf,TRAINING\nCERTIFICATIONS\nJessica Claire\n@ 10...,shell scripting. Extensive experience in softw...,NaN,SUMMARY
1,0(2).pdf,TRAINING\nFlorence Darlington Technical\nSchoo...,"Year Up - Information Technology\nLincoln, RI ...",+ Enterprise platforms\n+ Knowledge of Product...,JESSICA\nCLAIRE\n@ resumesample@example.com\n&...
2,0(3).pdf,"Caci International Inc. | Fort Detrick, MD\nEx...",NaN,"Ab-initio GDE 3.1.5, Co >Opsys (3.0 f02.15), T...",Jessica Claire\n@ 100 Montgomery St. 10th Floo...


---
# BAGIAN 2 — Cleaning & Standarisasi CV Kaggle

Bagian ini memproses data hasil OCR (Bagian 1) melalui empat sub-tahap:

| Sub-tahap | Proses |
|-----------|--------|
| **2A** | Standarisasi kolom `Pendidikan` — level pendidikan terstruktur |
| **2B** | Standarisasi kolom `Skills` — ekstraksi & normalisasi via kamus IT |
| **2C** | Standarisasi kolom `Pengalaman` — kategori lama pengalaman |
| **2D** | Penyempurnaan kolom `detail` via Groq API |

Kamus skill IT dimuat terlebih dahulu karena digunakan oleh sub-tahap 2B dan 2D.

## 2.0 Load Data & Kamus Skill

Data diambil dari `df_ocr` (hasil Bagian 1) atau langsung dari file xlsx
jika Bagian 1 dilewati.

Kamus skill dimuat dari `kamus_skill_cv.csv` dan dikompilasi menjadi
list regex pattern yang diurutkan dari terpanjang ke terpendek
agar `Spring Boot` selalu match sebelum `Spring`.

In [8]:
# LOAD DATA
df = pd.read_excel('../dataset_mentah/hasil_cv_kaggle_it_ocr.xlsx')
df = df.drop(columns=['Nama File'], errors='ignore')
print(f'✅ Shape awal: {df.shape}')
print('   Kolom:', df.columns.tolist())
df.head(3)

✅ Shape awal: (3402, 4)
   Kolom: ['Pendidikan', 'Pengalaman', 'Skills', 'Lainnya']


,Pendidikan,Pengalaman,Skills,Lainnya
0,TRAINING\nCERTIFICATIONS\nJessica Claire\n@ 10...,shell scripting. Extensive experience in softw...,NaN,SUMMARY
1,TRAINING\nFlorence Darlington Technical\nSchoo...,"Year Up - Information Technology\nLincoln, RI ...",+ Enterprise platforms\n+ Knowledge of Product...,JESSICA\nCLAIRE\n@ resumesample@example.com\n&...
2,"Caci International Inc. | Fort Detrick, MD\nEx...",NaN,"Ab-initio GDE 3.1.5, Co >Opsys (3.0 f02.15), T...",Jessica Claire\n@ 100 Montgomery St. 10th Floo...


In [9]:
# LOAD KAMUS SKILL
kamus_df = pd.read_csv('../kamus_skill_cv.csv')
print(f'✅ Kamus skill: {len(kamus_df):,} variasi → {kamus_df["formal"].nunique():,} skill formal')

SKILL_MAP: dict[str, str] = {
    row['variasi'].strip(): row['formal'].strip()
    for _, row in kamus_df.iterrows()
    if pd.notna(row['variasi']) and pd.notna(row['formal'])
}

# Regex diurutkan dari terpanjang ke terpendek
# agar 'Spring Boot' match sebelum 'Spring'
SKILL_PATTERNS: list[tuple[re.Pattern, str]] = []
for variasi, formal in sorted(SKILL_MAP.items(), key=lambda x: -len(x[0])):
    try:
        SKILL_PATTERNS.append((
            re.compile(rf'(?i)(?<![\w\-]){re.escape(variasi)}(?![\w\-])'),
            formal,
        ))
    except re.error:
        pass

print(f'✅ Total regex patterns siap: {len(SKILL_PATTERNS):,}')

✅ Kamus skill: 924 variasi → 787 skill formal
✅ Total regex patterns siap: 924


---
## BAGIAN 2A — Standarisasi Pendidikan

Kolom `Pendidikan` berisi teks bebas hasil OCR yang beragam formatnya.
Fungsi `extract_education_level()` menormalkan teks tersebut ke salah satu
dari enam level pendidikan standar menggunakan regex multi-pola.

Hierarki level yang dikenali (dari tertinggi ke terendah):

| Label Output | Kata Kunci / Pola |
|---|---|
| `PhD (S3)` | `phd`, `doctorate`, `d.sc` |
| `Magister (S2)` | `m.sc`, `m.eng`, `mba`, `master`, `postgraduate` |
| `Sarjana (S1)` | `b.sc`, `b.eng`, `bachelor`, `undergraduate`, `b.tech` |
| `Diploma 3 (D3)` | `associate degree`, `two-year college` |
| `Diploma 1/2 (D1/D2)` | `diploma`, `certificate`, `vocational` |
| `SMA/SMK` | `high school`, `secondary school`, `ged` |
| *(tidak ada)* | → `tidak ada pendidikan spesifik` |

In [10]:
def extract_education_level(text) -> str:
    """Normalisasi teks pendidikan bebas ke label level pendidikan standar."""
    if pd.isna(text):
        return 'tidak ada pendidikan spesifik'
    t = str(text).lower()
    checks = [
        ([r'ph\.?d', r'doctor(?:ate|al)?\s+(?:of|in|degree)', r'd\.?sc'], 'PhD (S3)'),
        ([r'm\.?s\.?c?\b', r'm\.?eng\b', r'm\.?b\.?a\b', r'm\.?a\.\b',
          r'master(?:s|\'s)?\s+(?:of|in|degree)', r'postgraduate'], 'Magister (S2)'),
        ([r'b\.?s\.?c?\b', r'b\.?eng\b', r'b\.?a\.\b', r'b\.?b\.?a\b',
          r'bachelor(?:s|\'s)?\s+(?:of|in|degree)', r'undergraduate',
          r'\b(?:b\.?tech|be\b|b\.e\.)', r'four.year\s+degree'], 'Sarjana (S1)'),
        ([r'associate(?:s|\'s)?\s+(?:of|in|degree)', r'two.year\s+(?:college|degree)',
          r'junior\s+college'], 'Diploma 3 (D3)'),
        ([r'diploma(?:\s+in)?', r'certif(?:icate|ication)\s+(?:in|of|program)',
          r'vocational', r'trade\s+school'], 'Diploma 1/2 (D1/D2)'),
        ([r'high\s+school', r'secondary\s+school', r'ged\b', r'senior\s+high'], 'SMA/SMK'),
    ]
    for patterns, label in checks:
        if any(re.search(p, t) for p in patterns):
            return label
    return 'tidak ada pendidikan spesifik'


# Preview distribusi pendidikan
print('Distribusi Level Pendidikan:')
print(df['Pendidikan'].apply(extract_education_level).value_counts().to_string())
print('\n✅ Fungsi standarisasi pendidikan siap.')

Distribusi Level Pendidikan:
Pendidikan
tidak ada pendidikan spesifik    1369
Magister (S2)                    1007
Sarjana (S1)                      823
Diploma 3 (D3)                     70
Diploma 1/2 (D1/D2)                65
PhD (S3)                           49
SMA/SMK                            19

✅ Fungsi standarisasi pendidikan siap.


---
## BAGIAN 2B — Standarisasi Skill

Fungsi `extract_skills()` mengekstrak semua skill dari kolom `Skills` dan `Pengalaman`
secara bersamaan menggunakan `SKILL_PATTERNS` yang sudah dikompilasi di Bagian 2.0.

Alur proses:
1. Gabungkan teks dari kolom `Skills` dan `Pengalaman`
2. Cocokkan setiap pattern regex ke teks gabungan
3. Kumpulkan semua skill formal yang ditemukan (tanpa duplikat)
4. Kembalikan sebagai list terurut alfabet

Setelah ekstraksi, baris yang **tidak memiliki skill sama sekali** dihapus
karena tidak informatif untuk proses lanjutan.

In [13]:
def extract_skills(row, cols=('Skills', 'Pengalaman')) -> list[str]:
    """Ekstrak skill dari kolom yang ditentukan menggunakan SKILL_PATTERNS"""
    combined = ' '.join(str(row.get(c, '')) for c in cols if pd.notna(row.get(c)))
    if not combined.strip():
        return []
    found: set[str] = set()
    for pattern, formal in SKILL_PATTERNS:
        if pattern.search(combined):
            found.add(formal)
    return sorted(found)


# Terapkan ke seluruh df
df['Skills_Extracted'] = df.apply(extract_skills, axis=1)

before = len(df)
df = df[df['Skills_Extracted'].apply(len) > 0].reset_index(drop=True)
after = len(df)

print(f'✅ Ekstraksi skill selesai.')
print(f'   Baris sebelum filter : {before:,}')
print(f'   Baris tanpa skill    : {before - after:,} (dihapus)')
print(f'   Baris tersisa        : {after:,}')

skill_counts = df['Skills_Extracted'].apply(len)
print(f'\nDistribusi jumlah skill per CV:')
print(f'   Rata-rata : {skill_counts.mean():.1f} skill')
print(f'   Min       : {skill_counts.min()} skill')
print(f'   Max       : {skill_counts.max()} skill')

✅ Ekstraksi skill selesai.
   Baris sebelum filter : 2,953
   Baris tanpa skill    : 0 (dihapus)
   Baris tersisa        : 2,953

Distribusi jumlah skill per CV:
   Rata-rata : 19.9 skill
   Min       : 1 skill
   Max       : 127 skill


---
## BAGIAN 2C — Standarisasi Pengalaman

Kolom `Pengalaman` berisi teks bebas yang perlu dinormalisasi ke kategori
lama pengalaman yang terstruktur.

Dua strategi ekstraksi lama pengalaman (dalam urutan prioritas):

1. **Eksplisit** : cari frasa seperti `"5+ years of experience"`, `"over 3 years"`, dll.
2. **Dari rentang tahun** : cari tahun-tahun di teks (mis. `2018–2023`) dan hitung selisihnya.
   Jika ada kata `present`/`current`, tahun akhir dianggap 2024.

Hasil konversi ke kategori:

| Rentang Tahun | Label |
|---|---|
| < 1 tahun | `< 1 tahun` |
| 1–2 tahun | `1-2 tahun` |
| 2–3 tahun | `2-3 tahun` |
| 3–5 tahun | `3-5 tahun` |
| 5–10 tahun | `5-10 tahun` |
| ≥ 10 tahun | `10+ tahun` |
| Tidak ditemukan | `tidak ada` |

In [15]:
def _extract_years_explicit(text: str):
    """Ekstrak lama pengalaman dari frasa eksplisit (mis. '5 years of experience')"""
    if pd.isna(text): return None
    t = str(text).lower()
    for p in [
        r'(\d+)\+?\s+years?\s+of\s+(?:professional\s+)?experience',
        r'(\d+)\+?\s+years?\s+experience', r'over\s+(\d+)\s+years?',
        r'(\d+)\s*\+\s*years?', r'more\s+than\s+(\d+)\s+years?',
        r'(\d+)\s+years?\s+(?:of\s+)?(?:work|professional|industry)',
    ]:
        m = re.search(p, t)
        if m: return float(m.group(1))
    return None


def _extract_years_from_dates(text: str):
    """Hitung lama pengalaman dari rentang tahun yang ditemukan di teks"""
    if pd.isna(text): return None
    years = [int(y) for y in re.findall(r'\b(19[89]\d|20[012]\d)\b', str(text))]
    if not years: return None
    has_present = bool(re.search(r'\b(current|present|ongoing|now)\b', str(text).lower()))
    end_year = 2024 if has_present else max(years)
    total = end_year - min(years)
    return max(0, total)


def _categorize_experience(years) -> str:
    """Konversi lama pengalaman (angka) ke label kategori"""
    if years is None or (isinstance(years, float) and np.isnan(years)): return 'tidak ada'
    if years < 1:  return '< 1 tahun'
    if years < 2:  return '1-2 tahun'
    if years < 3:  return '2-3 tahun'
    if years < 5:  return '3-5 tahun'
    if years < 10: return '5-10 tahun'
    return '10+ tahun'


def extract_experience_category(row) -> str:
    """Pipeline utama: gabungkan teks Pengalaman + Pendidikan lalu kategorikan"""
    combined = ' '.join([
        str(row.get('Pengalaman', '')) if pd.notna(row.get('Pengalaman')) else '',
        str(row.get('Pendidikan', ''))  if pd.notna(row.get('Pendidikan')) else '',
    ])
    years = _extract_years_explicit(combined) or _extract_years_from_dates(str(row.get('Pengalaman', '')))
    return _categorize_experience(years)


# Preview distribusi pengalaman
print('Distribusi Kategori Pengalaman:')
print(df.apply(extract_experience_category, axis=1).value_counts().to_string())
print('\n✅ Fungsi standarisasi pengalaman siap.')

Distribusi Kategori Pengalaman:
10+ tahun     1413
5-10 tahun     762
tidak ada      405
3-5 tahun      229
2-3 tahun       83
1-2 tahun       38
< 1 tahun       23

✅ Fungsi standarisasi pengalaman siap.


---
## BAGIAN 2D — Perbaikan Kolom `detail` via Groq API

Kolom `detail` diisi menggunakan LLM (LLaMA via Groq API).
Input adalah teks dari kolom `Lainnya` (bagian CV yang tidak masuk seksi manapun),
dan output adalah satu kalimat ringkasan pengalaman kerja terstruktur.

Fitur keamanan:
- **Checkpoint otomatis** setiap `BATCH_SIZE` baris — aman kalau koneksi putus
- **Retry otomatis** saat rate limit 429 — tunggu `RETRY_WAIT` detik lalu coba lagi
- **Validasi output** — output kosong / tidak informatif tidak disimpan

> Ganti nilai `CHECKPOINT` dengan nama file checkpoint lama untuk melanjutkan proses,
> atau nama file baru untuk memulai dari awal.

### 2D.1 Konfigurasi Groq

In [29]:
GROQ_API_KEY        = getpass.getpass('Input GROQ API KEY: ')
DELAY_BETWEEN_CALLS = 1.0
MAX_INPUT_CHARS     = 2000
MAX_RETRIES         = 2
RETRY_WAIT          = 30
MIN_OUTPUT_CHARS    = 20
MIN_INPUT_CHARS     = 3

NOINFO_PHRASES = [
    'no work experience', 'no experience', 'no information', 'there is no',
    'no job', 'cannot be determined', 'not available', 'no relevant',
    'none provided', 'not mentioned', 'n/a',
]

groq_client = Groq(api_key=GROQ_API_KEY)
print('✅ Groq client siap')

✅ Groq client siap


### 2D.2 Fungsi Groq

Tiga fungsi utama:
- `build_prompt()` : prompt instruksi untuk LLM
- `extract_detail_groq()` : panggil API dengan retry logic
- `run_groq()` : pipeline lengkap: prompt >> API >> normalisasi nama skill

In [16]:
def normalize_detail(text: str) -> str:
    """Normalisasi nama skill di output Groq menggunakan kamus formal"""
    if not text: return text
    for pattern, formal in SKILL_PATTERNS:
        text = pattern.sub(formal, text)
    return text


def build_prompt(lainnya_text: str):
    """Buat prompt dari teks kolom Lainnya; return None kalau teks terlalu pendek"""
    snippet = str(lainnya_text).strip()[:MAX_INPUT_CHARS] if pd.notna(lainnya_text) else ''
    if not snippet or len(snippet) < MIN_INPUT_CHARS: return None
    return f"""Kamu adalah pembersih data CV. Di bawah ini adalah teks dari kolom 'Lainnya' sebuah CV.

TUGAS:
Ekstrak dan tulis ulang detail pengalaman kerja dalam SATU kalimat panjang yang mengalir.
Struktur wajib: [skill teknis spesifik] + [lama & bidang pengalaman] + [tugas utama pekerjaan].
Gunakan kata kerja aktif: menguasai, mampu, berpengalaman, bertanggung jawab.
Bahasa Indonesia. Tanpa bullet. Tanpa kalimat pembuka/penutup. Tanpa enter. Langsung ke inti.
Jika teks sama sekali tidak mengandung informasi pekerjaan biarkan KOSONG.

TEKS CV:
{snippet}
"""


def is_valid_output(text: str) -> bool:
    """Validasi output Groq — tolak teks kosong atau tidak informatif"""
    if not text or len(text.strip()) < MIN_OUTPUT_CHARS: return False
    t = text.lower().strip()
    if t in ('skip', '') or t.startswith('[error'): return False
    return not any(p in t for p in NOINFO_PHRASES)


def extract_detail_groq(prompt: str, client) -> str:
    """Panggil Groq API dengan retry otomatis saat rate limit"""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = client.chat.completions.create(
                messages=[{'role': 'user', 'content': prompt}],
                model='llama-3.1-8b-instant',
                temperature=0.3, max_tokens=100,
            )
            result = response.choices[0].message.content.strip()
            if not is_valid_output(result):
                if attempt < MAX_RETRIES:
                    time.sleep(1); continue
                return ''
            return result
        except Exception as e:
            err = str(e)
            if '429' in err or 'rate_limit' in err.lower():
                print(f'  [Attempt {attempt}] Rate limit 429. Tunggu {RETRY_WAIT}s...')
                time.sleep(RETRY_WAIT)
            elif attempt < MAX_RETRIES:
                time.sleep(5)
            else:
                print(f'  Semua retry habis. Error: {err[:80]}')
                return ''
    return ''


def run_groq(text: str) -> str:
    """Pipeline utama: build prompt >> API call >> normalisasi skill"""
    prompt = build_prompt(text)
    if prompt is None: return ''
    return normalize_detail(extract_detail_groq(prompt, groq_client))


print('✅ Fungsi Groq siap.')

✅ Fungsi Groq siap.


### 2D.3 Uji Coba 3 Data

Uji pipeline Groq pada 3 baris sampel sebelum batch besar
untuk memastikan prompt dan koneksi API berfungsi.

In [17]:
print('UJI COBA 3 DATA')
for i, row in df.sample(3, random_state=50).iterrows():
    teks  = row.get('Lainnya', '')
    hasil = run_groq(teks)
    print(f'[{i}] Input  : {repr(str(teks)[:80])}')
    print(f'     Output : {hasil}')
    print()
    time.sleep(DELAY_BETWEEN_CALLS)

UJI COBA 3 DATA
[2053] Input  : 'JASMINE BELL\nBlockchain Developer\n\\ +1-5491-295-157 @ ronnyphoenix@gmail.com\n@ h'
     Output : Menguasai JavaScript, Go, dan Python selama 10 tahun dalam bidang Software Engineering, dengan pengalaman bertanggung jawab sebagai co-founder ConanSwap, sebuah DEX dengan 20.000 pendukung aktif, dan memiliki pengetahuan luas dalam kontrak cerdas, algoritma kesepakatan, dan struktur data.

[1670] Input  : 'JESSICA CLAIRE\nMontgomery Street, San Francisco, CA 94105\n(555) 432-1000 - resum'
     Output : Menguasai pengelolaan database performa dan upgrade untuk meningkatkan efisiensi dan mencapai tujuan korporat. Berpengalaman selama 5 tahun dalam mengelola AWS, termasuk Amazon EC2, VPC, EBS, ELB, Cloud Front, LAM, RDS, Cloud Watch, Auto Scaling, Cloud Formation, Docker, dan Cloud Endure. Mampu merancang dan mengembangkan aplikasi web

[12] Input  : 'ORLANDO CAMPA\nEntry Level Business Analyst: Analyzing Data to Discover Business-'
     Output : Kamu mampu 

### 2D.4 Batch Processing dengan Checkpoint

> Ganti `CHECKPOINT` ke nama file lama untuk lanjut, atau nama baru untuk mulai ulang.
> Ubah `SAMPLE_N = None` untuk proses semua data.

In [16]:
CHECKPOINT = '../checkpoint_LLM/detail_checkpoint_dataset_cv_ocr.xlsx'
BATCH_SIZE = 20
SAMPLE_N   = None

df_run = df.head(SAMPLE_N) if SAMPLE_N else df

# Backup kolom asli (sekali saja)
if 'detail_asli' not in df.columns:
    df['detail_asli'] = df.get('detail', pd.Series([''] * len(df), index=df.index))

# Load checkpoint jika ada
if os.path.exists(CHECKPOINT):
    ckpt         = pd.read_excel(CHECKPOINT)
    ckpt         = ckpt.drop_duplicates(subset='original_index', keep='last')
    done_indices = set(ckpt['original_index'].astype(int).tolist())
    print(f'Checkpoint ditemukan')
    print(f'Total selesai : {len(done_indices):,} baris')
    print(f'Melanjutkan proses...\n')
else:
    ckpt         = pd.DataFrame(columns=['original_index', 'detail'])
    done_indices = set()
    print('Belum ada checkpoint, mulai dari awal.\n')

total   = len(df_run)
pending = [idx for idx in df_run.index if idx not in done_indices]
new_rows, n_skip, n_error = [], 0, 0

print('=' * 60)
print(f'Total data      : {total:,}')
print(f'Sudah diproses  : {len(done_indices):,}')
print(f'Sisa data       : {len(pending):,}')
print(f'Batch size      : {BATCH_SIZE}')
print('=' * 60)

for count, idx in enumerate(pending, start=1):
    row  = df_run.loc[idx]
    teks = row.get('Lainnya', '')

    processed = len(done_indices) + len(new_rows)
    if count % 5 == 0 or count == 1:
        pct = (processed / total) * 100
        print(f'[{processed:>5}/{total}] ({pct:.1f}%)')

    if not pd.notna(teks) or len(str(teks).strip()) < MIN_INPUT_CHARS:
        n_skip += 1
        new_rows.append({'original_index': idx, 'detail': ''})
        continue

    try:
        result = run_groq(teks)
        if not result:
            n_error += 1
        new_rows.append({'original_index': idx, 'detail': result})
    except Exception as e:
        n_error += 1
        print(f'Error index={idx} | {str(e)}')
        new_rows.append({'original_index': idx, 'detail': ''})

    if len(new_rows) >= BATCH_SIZE:
        batch_df     = pd.DataFrame(new_rows)
        ckpt         = pd.concat([ckpt, batch_df], ignore_index=True)
        ckpt         = ckpt.drop_duplicates(subset='original_index', keep='last')
        ckpt.to_excel(CHECKPOINT, index=False)
        done_indices.update(batch_df['original_index'].tolist())
        print(f'  Checkpoint disimpan | Total saved: {len(done_indices):,}')
        new_rows = []

    time.sleep(DELAY_BETWEEN_CALLS)

if new_rows:
    batch_df     = pd.DataFrame(new_rows)
    ckpt         = pd.concat([ckpt, batch_df], ignore_index=True)
    ckpt         = ckpt.drop_duplicates(subset='original_index', keep='last')
    ckpt.to_excel(CHECKPOINT, index=False)
    done_indices.update(batch_df['original_index'].tolist())
    print(f'  Checkpoint terakhir disimpan | Total saved: {len(done_indices):,}')

# Gabungkan hasil ke df
ckpt_indexed = (
    ckpt
    .drop_duplicates(subset='original_index', keep='last')
    .set_index('original_index')['detail']
)
df['detail'] = df.index.to_series().map(ckpt_indexed).fillna('')

filled = (df['detail'] != '').sum()
print(f'\n{"=" * 60}')
print('✅ SELESAI!')
print(f'{"=" * 60}')
print(f'Total data        : {total:,}')
print(f'Berhasil diproses : {len(done_indices):,}')
print(f'Hasil kosong/skip : {n_skip:,}')
print(f'Error             : {n_error:,}')
print(f'Kolom terisi      : {filled:,}')

Checkpoint ditemukan
Total selesai : 2,953 baris
Melanjutkan proses...

Total data      : 2,953
Sudah diproses  : 2,953
Sisa data       : 0
Batch size      : 20

✅ SELESAI!
Total data        : 2,953
Berhasil diproses : 2,953
Hasil kosong/skip : 0
Error             : 0
Kolom terisi      : 2,310


### 2D.5 Susun `df_final`

Gabungkan hasil standarisasi 2A, 2B, 2C, dan 2D ke dalam satu DataFrame final.

In [34]:
df_final = pd.DataFrame({
    'pendidikan': df['Pendidikan'].apply(extract_education_level),
    'pengalaman': df.apply(extract_experience_category, axis=1),
    'skill'     : df['Skills_Extracted'].apply(lambda x: ', '.join(x) if x else ''),
    'detail'    : df['detail'],
})

print(f'✅ df_final tersusun. Shape: {df_final.shape}')
print('   Kolom:', df_final.columns.tolist())
df_final.sample(5)

✅ df_final tersusun. Shape: (2953, 4)
   Kolom: ['pendidikan', 'pengalaman', 'skill', 'detail']


,pendidikan,pengalaman,skill,detail
2194,Sarjana (S1),5-10 tahun,"Adobe Creative Suite, CSS3, HTMLS5, JavaScript...",Menguasai desain web dengan pengalaman 6+ tahu...
897,Magister (S2),10+ tahun,"BI, CRM, Customer Service, FI, HP, HR, Help De...",Menguasai SAP Security dengan GRC 5.3/GRC AC 1...
2426,tidak ada pendidikan spesifik,tidak ada,Digital Media,"Menguasai Avid, FCP 7/X, Premiere Pro, Adobe P..."
799,Sarjana (S1),10+ tahun,Creative Writing,"Menguasai penjualan, berpengalaman 10 tahun da..."
1712,Diploma 1/2 (D1/D2),tidak ada,"Manual Testing, Microsoft Excel, Networking, P...","Menguasai teknologi otomotif, berpengalaman 0 ..."


---
# BAGIAN 3 — Cleaning Dataset Tambahan (Kaggle Resume)

Bagian ini memproses dataset tambahan dari Kaggle yang berformat teks resume panjang
(satu kolom `Resume` berisi seluruh isi CV).

Perbedaan dengan Bagian 2:
- Pendidikan & pengalaman diekstrak langsung dari kolom `Resume`
- Skill diekstrak dari kolom `Resume` (bukan `Skills` + `Pengalaman`)
- Prompt Groq menggunakan pre-filter untuk menghemat token (ekstrak bagian
  *Summary* atau *Experience* terlebih dahulu sebelum dikirim ke API)

## 3.1 Load Dataset Tambahan

In [18]:
df_tambahan = pd.read_excel('..\dataset_mentah\dataset_tambahan_cv.xlsx')
print(f'✅ Shape awal dataset tambahan: {df_tambahan.shape}')
print('   Kolom:', df_tambahan.columns.tolist())
df_tambahan.head(3)

✅ Shape awal dataset tambahan: (10174, 8)
   Kolom: ['ID', 'Name', 'Role', 'Transcript', 'Resume', 'decision', 'Reason_for_decision', 'Job_Description']


,ID,Name,Role,Transcript,Resume,decision,Reason_for_decision,Job_Description
0,jasojo159,Jason Jones,E-commerce Specialist,"Interviewer: Good morning, Jason. It's great t...",Here's a professional resume for Jason Jones:\...,reject,Lacked leadership skills for a senior position.,Be part of a passionate team at the forefront ...
1,annma759,Ann Marshall,Game Developer,Interview Scene\n\nA conference room with a ta...,Here's a professional resume for Ann Marshall:...,select,Strong technical skills in AI and ML.,Help us build the next-generation products as ...
2,patrmc729,Patrick Mcclain,Human Resources Specialist,Interview Setting: A conference room in a medi...,Here's a professional resume for Patrick Mccla...,reject,Insufficient system design expertise for senio...,We need a Human Resources Specialist to enhanc...


## 3.2 Standarisasi Pendidikan (Dataset Tambahan)

Fungsi yang sama dengan Bagian 2A, tetapi membaca dari kolom `Resume`.

In [23]:
def extract_education_tambahan(text) -> str:
    """Normalisasi pendidikan dari teks kolom Resume (dataset tambahan)"""
    return extract_education_level(text)   # reuse fungsi 2A


# Preview distribusi pendidikan
print('Distribusi Pendidikan (dataset tambahan):')
print(df_tambahan['Resume'].apply(extract_education_tambahan).value_counts().to_string())
print('\n✅ Fungsi pendidikan tambahan siap.')

Distribusi Pendidikan (dataset tambahan):
Resume
Magister (S2)                    9337
PhD (S3)                          475
tidak ada pendidikan spesifik     214
Sarjana (S1)                      136
SMA/SMK                            12

✅ Fungsi pendidikan tambahan siap.


## 3.3 Standarisasi Skill (Dataset Tambahan)

Skill diekstrak dari kolom `Resume` menggunakan `SKILL_PATTERNS` yang sama.

In [24]:
def extract_skills_tambahan(row, cols=('Resume',)) -> list[str]:
    """Ekstrak skill dari kolom Resume (dataset tambahan)"""
    combined = ' '.join(str(row.get(c, '')) for c in cols if pd.notna(row.get(c)))
    if not combined.strip(): return []
    found: set[str] = set()
    for pattern, formal in SKILL_PATTERNS:
        if pattern.search(combined): found.add(formal)
    return sorted(found)


df_tambahan['Skills_Extracted'] = df_tambahan.apply(extract_skills_tambahan, axis=1)

before = len(df_tambahan)
df_tambahan = df_tambahan[df_tambahan['Skills_Extracted'].apply(len) > 0].reset_index(drop=True)
after = len(df_tambahan)

print(f'✅ Ekstraksi skill tambahan selesai.')
print(f'   Baris sebelum filter : {before:,}')
print(f'   Baris tanpa skill    : {before - after:,} (dihapus)')
print(f'   Baris tersisa        : {after:,}')

✅ Ekstraksi skill tambahan selesai.
   Baris sebelum filter : 10,174
   Baris tanpa skill    : 9 (dihapus)
   Baris tersisa        : 10,165


## 3.4 Standarisasi Pengalaman (Dataset Tambahan)

Pengalaman diekstrak langsung dari teks kolom `Resume`.

In [25]:
def extract_experience_tambahan(resume: str) -> str:
    """Ekstrak kategori pengalaman dari teks kolom Resume"""
    if pd.isna(resume): return 'tidak ada'
    text  = str(resume)
    years = _extract_years_explicit(text) or _extract_years_from_dates(text)
    return _categorize_experience(years)


# Preview distribusi pengalaman
print('Distribusi Pengalaman (dataset tambahan):')
print(df_tambahan['Resume'].apply(extract_experience_tambahan).value_counts().to_string())
print('\n✅ Fungsi pengalaman tambahan siap.')

Distribusi Pengalaman (dataset tambahan):
Resume
5-10 tahun    7074
10+ tahun     1353
tidak ada     1042
3-5 tahun      573
2-3 tahun       79
1-2 tahun       29
< 1 tahun       15

✅ Fungsi pengalaman tambahan siap.


## 3.5 Perbaikan `detail` via Groq (Dataset Tambahan)

Prompt tambahan menggunakan pre-filter `prefilter_resume()` yang mengekstrak
bagian *Summary* atau *Experience* dari teks resume sebelum dikirim ke LLM,
sehingga token yang digunakan lebih efisien.

> Ganti `CHECKPOINT_TAMBAHAN` ke nama file lama untuk lanjut, atau nama baru untuk mulai ulang.
> Ubah `SAMPLE_N_TB = None` untuk proses semua data.

### 3.5.1 Konfigurasi Groq Tambahan

In [27]:
GROQ_API_KEY_TAMBAHAN = getpass.getpass('Input GROQ API KEY (tambahan): ')
groq_client_tambahan  = Groq(api_key=GROQ_API_KEY_TAMBAHAN)
print('✅ Groq client tambahan siap')

✅ Groq client tambahan siap


### 3.5.2 Pre-filter & Fungsi Prompt

In [30]:
# Header seksi yang dicari untuk pre-filter
_SUMMARY_HEADERS = [
    'summary', 'professional summary', 'career summary', 'career profile',
    'executive summary', 'objective', 'career objective', 'profile', 'about me',
]
_EXPERIENCE_HEADERS = [
    'work experience', 'experience', 'professional experience',
    'employment history', 'work history',
]
_STOP_HEADERS = re.compile(
    r'^\s*(?:education|skills|technical skills|projects|certifications|'
    r'awards|publications|references|languages|interests)\b', re.IGNORECASE
)


def _extract_section(text: str, headers: list[str], max_chars=MAX_INPUT_CHARS) -> str:
    """Ekstrak satu seksi dari teks resume berdasarkan daftar header"""
    lines     = str(text).splitlines()
    header_re = re.compile(
        r'^\s*(?:' + '|'.join(re.escape(h) for h in headers) + r')\s*[:\-]?\s*(.*)$',
        re.IGNORECASE,
    )
    collecting, buf, char_count = False, [], 0
    for line in lines:
        if not collecting:
            m = header_re.match(line)
            if m:
                collecting = True
                inline = m.group(1).strip()
                if inline: return inline[:max_chars]
        else:
            if _STOP_HEADERS.match(line): break
            buf.append(line)
            char_count += len(line)
            if char_count >= max_chars: break
    return ' '.join(buf).strip()[:max_chars]


def prefilter_resume(resume_text: str) -> str:
    """Ambil bagian Summary / Experience dari resume untuk menghemat token"""
    if pd.isna(resume_text) or not str(resume_text).strip(): return ''
    text = str(resume_text)
    for headers in (_SUMMARY_HEADERS, _EXPERIENCE_HEADERS):
        snippet = _extract_section(text, headers)
        if snippet and len(snippet) >= MIN_INPUT_CHARS: return snippet
    return text.strip()[:MAX_INPUT_CHARS]


def build_prompt_tambahan(resume_text: str):
    """Buat prompt khusus dataset tambahan dengan pre-filter resume"""
    snippet = prefilter_resume(resume_text)
    if not snippet or len(snippet) < MIN_INPUT_CHARS: return None
    return f"""Kamu adalah pembersih data CV. Di bawah ini adalah penggalan teks CV
(sudah dipotong ke bagian paling relevan).

TUGAS:
Ekstrak dan tulis ulang detail pengalaman kerja dalam SATU kalimat panjang yang mengalir.
Struktur wajib: [skill teknis spesifik] + [lama & bidang pengalaman] + [tugas utama pekerjaan].
Gunakan kata kerja aktif: menguasai, mampu, berpengalaman, bertanggung jawab.
Bahasa Indonesia. Tanpa bullet. Tanpa enter. Tanpa kalimat pembuka/penutup.
Jika data tidak cukup → tulis: NULL

TEKS CV:
{snippet}
"""


def run_groq_tambahan(resume_text: str) -> str:
    """Pipeline Groq untuk dataset tambahan."""
    prompt = build_prompt_tambahan(resume_text)
    if prompt is None: return ''
    return normalize_detail(extract_detail_groq(prompt, groq_client_tambahan))


print('✅ Fungsi Groq tambahan siap.')

✅ Fungsi Groq tambahan siap.


### 3.5.3 Uji Coba 3 Data Tambahan

In [29]:
print('UJI COBA 3 DATA TAMBAHAN')
for i, row in df_tambahan.sample(3, random_state=42).iterrows():
    teks  = row.get('Resume', '')
    hasil = run_groq_tambahan(teks)
    print(f'[{i}] Input  : {repr(str(teks)[:80])}')
    print(f'     Output : {hasil}')
    print()
    time.sleep(DELAY_BETWEEN_CALLS)

UJI COBA 3 DATA TAMBAHAN
[3065] Input  : "Here's a sample professional resume for Richard White:\n\nRichard White\nCloud Arch"
     Output : Berikut adalah ekstrak dan tulisan ulang detail pengalaman kerja dalam satu kalimat panjang yang mengalir:

Menguasai teknologi cloud seperti AWS), Microsoft Microsoft Azure, dan platform lainnya, dengan pengalaman kerja lebih dari 8 tahun dalam merancang, mengimplementasikan, dan mengelola infrastruktur cloud skala besar, serta berpengalaman dalam menggunakan alat DevOps seperti Terraform, Kubernetes,

[9082] Input  : '**candidate profile: vanshika reddy**\n\n**interview summary:**\n\nvanshika reddy, a'
     Output : Menguasai Python dan R, 3 tahun pengalaman di bidang Data Analysis, bertanggung jawab untuk melakukan data wrangling, mampu melakukan Data Data Visualization dengan menggunakan Tableau, berpengalaman dalam melakukan analisis statistik dasar, NULL dalam pengalaman bekerja dengan Big Data dan Machine Learning.

[3095] Input  : 'Here is

### 3.5.4 Batch Processing dengan Checkpoint

In [31]:
CHECKPOINT_TAMBAHAN = '..\checkpoint_LLM\detail_checkpoint_dataset_tambahan_cv.xlsx'
BATCH_SIZE_TB       = 20
SAMPLE_N_TB         = None

df_run_tb = df_tambahan.head(SAMPLE_N_TB) if SAMPLE_N_TB else df_tambahan

# Load checkpoint jika ada
if os.path.exists(CHECKPOINT_TAMBAHAN):
    ckpt_tb = pd.read_excel(CHECKPOINT_TAMBAHAN)
    ckpt_tb = ckpt_tb.drop_duplicates(subset='original_index', keep='last')
    done_tb = set(ckpt_tb['original_index'].astype(int).tolist())
    print(f'Checkpoint ditemukan')
    print(f'Melanjutkan proses...\n')
else:
    ckpt_tb = pd.DataFrame(columns=['original_index', 'detail'])
    done_tb = set()
    print('Belum ada checkpoint, mulai dari awal.\n')

total_tb   = len(df_run_tb)
pending_tb = [idx for idx in df_run_tb.index if idx not in done_tb]
new_rows, n_skip, n_error = [], 0, 0

print('=' * 60)
print(f'Total data      : {total_tb:,}')
print(f'Sudah diproses  : {total_tb - len(pending_tb):,}')
print(f'Sisa data       : {len(pending_tb):,}')
print(f'Batch size      : {BATCH_SIZE_TB}')
print('=' * 60)

for count, idx in enumerate(pending_tb, start=1):
    row  = df_run_tb.loc[idx]
    teks = row.get('Resume', '')

    processed = len(done_tb) + len(new_rows)
    if count % 5 == 0 or count == 1:
        pct = (processed / total_tb) * 100
        print(f'[{processed:>5}/{total_tb}] ({pct:.1f}%)')

    if not pd.notna(teks) or len(str(teks).strip()) < MIN_INPUT_CHARS:
        n_skip += 1
        new_rows.append({'original_index': idx, 'detail': ''})
        continue

    try:
        result = run_groq_tambahan(teks)
        if not result:
            n_error += 1
        new_rows.append({'original_index': idx, 'detail': result})
    except Exception as e:
        n_error += 1
        print(f'Error index={idx} | {str(e)}')
        new_rows.append({'original_index': idx, 'detail': ''})

    if len(new_rows) >= BATCH_SIZE_TB:
        batch_df = pd.DataFrame(new_rows)
        ckpt_tb  = pd.concat([ckpt_tb, batch_df], ignore_index=True)
        ckpt_tb  = ckpt_tb.drop_duplicates(subset='original_index', keep='last')
        ckpt_tb.to_excel(CHECKPOINT_TAMBAHAN, index=False)
        done_tb.update(batch_df['original_index'].tolist())
        print(f'  Checkpoint disimpan | Total saved: {len(done_tb):,}')
        new_rows = []

    time.sleep(DELAY_BETWEEN_CALLS)

if new_rows:
    batch_df = pd.DataFrame(new_rows)
    ckpt_tb  = pd.concat([ckpt_tb, batch_df], ignore_index=True)
    ckpt_tb  = ckpt_tb.drop_duplicates(subset='original_index', keep='last')
    ckpt_tb.to_excel(CHECKPOINT_TAMBAHAN, index=False)
    done_tb.update(batch_df['original_index'].tolist())
    print(f'  Checkpoint terakhir disimpan | Total saved: {len(done_tb):,}')

# Gabungkan hasil ke df_tambahan
ckpt_tb_indexed = (
    ckpt_tb
    .drop_duplicates(subset='original_index', keep='last')
    .set_index('original_index')['detail']
)
df_tambahan['detail'] = df_tambahan.index.to_series().map(ckpt_tb_indexed).fillna('')

filled = (df_tambahan['detail'] != '').sum()
print(f'\n{"=" * 60}')
print('✅ SELESAI tambahan!')
print(f'{"=" * 60}')
print(f'Total data        : {total_tb:,}')
print(f'Berhasil diproses : {total_tb - len(pending_tb):,}')
print(f'Hasil kosong/skip : {n_skip:,}')
print(f'Error             : {n_error:,}')
print(f'Kolom terisi      : {filled:,}')

Checkpoint ditemukan
Melanjutkan proses...

Total data      : 10,165
Sudah diproses  : 10,165
Sisa data       : 0
Batch size      : 20

✅ SELESAI tambahan!
Total data        : 10,165
Berhasil diproses : 10,165
Hasil kosong/skip : 0
Error             : 0
Kolom terisi      : 10,026


## 3.6 Susun `df_final_tambahan`

In [32]:
df_final_tambahan = pd.DataFrame({
    'pendidikan': df_tambahan['Resume'].apply(extract_education_tambahan),
    'pengalaman': df_tambahan['Resume'].apply(extract_experience_tambahan),
    'skill'     : df_tambahan['Skills_Extracted'].apply(lambda x: ', '.join(x) if x else ''),
    'detail'    : df_tambahan['detail'],
})

print(f'✅ df_final_tambahan tersusun. Shape: {df_final_tambahan.shape}')
df_final_tambahan.head(5)

✅ df_final_tambahan tersusun. Shape: (10165, 4)


,pendidikan,pengalaman,skill,detail
0,Magister (S2),5-10 tahun,"Customer Service, Data Analysis, Facebook Ads,...",Menguasai sistem Inventory Management dan anal...
1,tidak ada pendidikan spesifik,5-10 tahun,"GitHub, Unity, Unreal Engine","menguasai Unity dan Unreal Engine, berpengalam..."
2,Magister (S2),5-10 tahun,"BambooHR, Conflict Resolution, Data Analysis, ...","menguasai HRIS sistem, recruitment marketing, ..."
3,Magister (S2),5-10 tahun,"Customer Service, Data Analysis, Google Analyt...","Menguasai SEO, mampu mengelola produk 10.000+,..."
4,Magister (S2),5-10 tahun,"Customer Service, Data Analysis, Google Analyt...","menguasai sistem manajemen inventori, 5+ tahun..."


---
# BAGIAN 4 — Gabungkan & Bersihkan Kolom `detail`

Dua dataset (`df_final` dari CV Kaggle dan `df_final_tambahan` dari Resume Kaggle)
digabung menjadi satu, lalu kolom `detail` dibersihkan dari:

1. **Frasa tidak-informatif** : seperti *"tidak ada informasi"*, *"NULL, NULL, NULL"*
2. **Kata pengantar di awal kalimat** : seperti *"Pengalaman kerja: ..."* → potong sebelum `:`
3. **NULL di tengah kalimat** : ganti dengan angka `0`

## 4.1 Gabungkan Dua Dataset

In [35]:
df_gabung = pd.concat([df_final, df_final_tambahan], ignore_index=True)
print(f'✅ Shape setelah digabung: {df_gabung.shape}')
print('   Kolom:', df_gabung.columns.tolist())
print(f'\nDistribusi Pendidikan (gabungan):')
print(df_gabung['pendidikan'].value_counts().to_string())

✅ Shape setelah digabung: (13118, 4)
   Kolom: ['pendidikan', 'pengalaman', 'skill', 'detail']

Distribusi Pendidikan (gabungan):
pendidikan
Magister (S2)                    10162
tidak ada pendidikan spesifik     1365
Sarjana (S1)                       924
PhD (S3)                           512
Diploma 3 (D3)                      64
Diploma 1/2 (D1/D2)                 63
SMA/SMK                             28


## 4.2 Bersihkan Kolom `detail`

Tiga regex pattern untuk membersihkan output LLM yang tidak bersih:
- `KOSONG_RE` : deteksi frasa tidak-informatif
- `PENGANTAR_RE` : potong kata pengantar sebelum `:`
- `NULL_MID_RE` : ganti NULL di tengah kalimat

In [36]:
KOSONG_RE    = re.compile(
    r'(?i)'
    r'tidak mengandung informasi'
    r'|tidak menemukan informasi'
    r'|kurang informasi'
    r'|tidak ada informasi'
    r'|(?:nul[\s,]*){2,}'
)
PENGANTAR_RE = re.compile(r'^[^:\d/(\n]{5,120}:\s?')
NULL_MID_RE  = re.compile(r'\bNULL\b')


def clean_detail(val) -> str:
    """Bersihkan satu nilai kolom detail dari noise LLM"""
    if pd.isna(val): return ''
    text = str(val).strip()
    if KOSONG_RE.search(text): return ''
    lines = []
    for line in text.split('\n'):
        line = PENGANTAR_RE.sub('', line).strip()
        if line: lines.append(line)
    text = ' '.join(lines).strip()
    if KOSONG_RE.search(text): return ''
    return NULL_MID_RE.sub('0', text).strip()


df_gabung['detail'] = df_gabung['detail'].apply(clean_detail)

total  = len(df_gabung)
kosong = (df_gabung['detail'] == '').sum()
print(f'✅ Pembersihan detail selesai.')
print(f'   Total baris         : {total:,}')
print(f'   Detail dikosongkan  : {kosong:,} ({kosong/total:.1%})')
print(f'   Detail terisi       : {total - kosong:,} ({(total - kosong)/total:.1%})')

sisa = df_gabung['detail'][df_gabung['detail'].str.contains(KOSONG_RE, na=False)]
print(f'\n   Sisa frasa tidak-informatif: {len(sisa)} baris')

✅ Pembersihan detail selesai.
   Total baris         : 13,118
   Detail dikosongkan  : 935 (7.1%)
   Detail terisi       : 12,183 (92.9%)

   Sisa frasa tidak-informatif: 0 baris


---
# BAGIAN 5 — Deduplikasi & Simpan Output Final

Dari setiap grup baris yang memiliki nilai `skill` identik,
pilih satu baris dengan kolom `detail` **terpanjang** (paling informatif).
Hasilnya disimpan sebagai file Excel final.

## 5.1 Cek Sebelum Deduplikasi

In [37]:
print(f'Shape sebelum deduplikasi : {df_gabung.shape}')
print(f'Duplikat (berdasar skill) : {df_gabung.duplicated(subset=["skill"]).sum():,}')
print(f'Unique skill              : {df_gabung["skill"].nunique():,}')

Shape sebelum deduplikasi : (13118, 4)
Duplikat (berdasar skill) : 739
Unique skill              : 12,379


## 5.2 Deduplikasi

Strategi: sort descending berdasarkan panjang `detail`,
lalu `drop_duplicates(keep='first')` agar baris dengan `detail` terpanjang yang dipertahankan.

In [ ]:
df_gabung['_detail_len'] = df_gabung['detail'].fillna('').str.len()

df_dedup = (
    df_gabung
    .sort_values('_detail_len', ascending=False)
    .drop_duplicates(subset=['skill'], keep='first')
    .drop(columns=['_detail_len'])
    .reset_index(drop=True)
)

df_dedup = df_dedup[['pendidikan', 'pengalaman', 'skill', 'detail']]

print(f'✅ Deduplikasi selesai.')
print(f'   Shape sebelum : {df_gabung.shape}')
print(f'   Shape sesudah : {df_dedup.shape}')
print(f'   Baris dihapus : {len(df_gabung) - len(df_dedup):,}')
print(f'   Duplikat sisa : {df_dedup.duplicated(subset=["skill"]).sum()}')

pd.set_option('display.max_colwidth', 200)
df_dedup.head(10)

✅ Deduplikasi selesai.
   Shape sebelum : (13118, 5)
   Shape sesudah : (12379, 4)
   Baris dihapus : 739
   Duplikat sisa : 0


,pendidikan,pengalaman,skill,detail
0,Magister (S2),5-10 tahun,"AWS, AWS Glue, Amazon Redshift, Amazon S3, Apache Beam, Apache Flink, Apache HTTP Server, Apache NiFi, Apache Spark, Big Data, Cloud Computing, Computer Science, Data Integration, Data Pipelines, ...","menguasai Python, Java, Scala, SQL, Hadoop, Apache HTTP Server Apache Spark, Amazon Amazon Redshift, Google Google BigQuery, Apache HTTP Server Beam, Apache HTTP Server Apache NiFi, AWS AWS Glue, ..."
1,Magister (S2),5-10 tahun,"AWS, Apache Beam, Apache Cassandra, Apache Flink, Apache HTTP Server, Apache Hive, Apache Pig, Apache Spark, BI, Big Data, Business Intelligence, Cloud Computing, Computer Science, D3.js, Data Int...","menguasai Python, Java, Scala, SQL, serta teknologi Big Data seperti Hadoop, Apache Spark, Apache Hive, Apache Pig; mampu merancang dan mengembangkan sistem data skala besar; bertanggung jawab dal..."
2,tidak ada pendidikan spesifik,10+ tahun,"Assembly, Business Analysis, Business Requirements, Change Management, ERP, FICO, Information Systems, Inventory Management, MM, PP, Relationship Management, SAP, SD, SRM, Supply Chain, System Int...","Menguasai Strategic Planning e Business-System Readiness, berpengalaman 10 tahun dalam bidang ERP, bertanggung jawab dalam Business Relationship Management, Training dan Change Management, Strateg..."
3,Sarjana (S1),10+ tahun,"Agile, Agile Methodology, Automation, BI, DBA, Disaster Recovery, Firewall, Query Analyzer, Query Optimization, Reporting, SQL, SQL Server, SSMS, SSRS, Statistics, Stored Procedures, T-SQL","menguasai SQL Server 2000-2017, berpengalaman 6 tahun di bidang database dengan fokus pada Performance Tuning, capacity planning, High Availability Solution, Disaster Recovery planning, serta meng..."
4,Magister (S2),5-10 tahun,"AWS, AWS Lambda, Agile, Agile Methodology, Amazon CloudWatch, Apache Cassandra, Azure Functions, C#, Cloud Computing, Cloud Cost Optimization, Cloud Functions, CloudTrail, Computer Science, Google...","Menguasai berbagai platform cloud, termasuk AWS, Microsoft Azure, dan Google Cloud Platform, dengan pengalaman 5+ tahun dalam Cloud Cost Optimization, Serverless Architecture, dan Cloud Computing,..."
5,Magister (S2),2-3 tahun,"API, AWS, Agile, Agile Methodology, Angular, Babel, CI/CD, CSS, Computer Science, Continuous Deployment, Database Management, DevOps, Django, Express.js, Git, GitHub, Google Cloud Platform, HTML, ...","Menguasai JavaScript, HTML/CSS, dan Python serta berpengalaman menggunakan framework React, Express.JavaScript.JavaScript, dan database MongoDB, MySQL. Bertanggung jawab mengembangkan aplikasi web..."
6,Magister (S2),5-10 tahun,"AWS, Apache Airflow, Apache Cassandra, Apache HTTP Server, Apache Kafka, Apache NiFi, Apache Spark, BI, Computer Science, D3.js, Data Analysis, Data Pipelines, Data Quality, Data Visualization, De...","Menguasai Python, R, SQL, dan Julia untuk analisis data dan pengembangan model Machine Learning selama 8+ tahun dalam bidang data science, bertanggung jawab untuk mengembangkan model prediktif unt..."
7,Magister (S2),tidak ada,"AWS, AWS Glue, Amazon Redshift, Apache Beam, Apache Cassandra, Apache HTTP Server, Apache Spark, Azure Data Factory, Big Data, Data Analysis, Data Management, Data Pipelines, Data Quality, Data Wa...","Menguasai Data Warehousing, ETL, Big Data technologies, dan Database Administration selama 5 tahun di bidang logistik, finance, dan healthcare, bertanggung jawab untuk merancang dan mengimplementa..."
8,Magister (S2),tidak ada,"AWS, AWS IAM, AWS Lambda, Automation, Azure Functions, Bash, CD, CI/CD, Cloud Computing, Cloud Cost Optimization, Cloud Functions, Computer Science, DevOps, Disaster Recovery, Docker, Firewall, Go...","Menguasai Cloud Cost Optimization, 8 tahun pengalaman di bidang Cloud Engineering, bertanggung jawab dalam merancang dan mengimplementasikan infrastruktur cloud untuk aplikasi skala besar mengguna..."
9,Magister (S2),tidak ada,"Active Directory, C++, Linux, MS, Microsoft Azure, Microsof

## 5.3 Simpan Output Final

In [39]:
OUTPUT_FINAL = '..\dataset_clean\dataset_cv.xlsx'
df_dedup.to_excel(OUTPUT_FINAL, index=False)

print(f'✅ Tersimpan: {OUTPUT_FINAL}')
print(f'   Shape   : {df_dedup.shape}')
print(f'\nPreview 5 baris pertama:')
df_dedup.head()

✅ Tersimpan: ..\dataset_clean\dataset_cv.xlsx
   Shape   : (12379, 4)

Preview 5 baris pertama:


,pendidikan,pengalaman,skill,detail
0,Magister (S2),5-10 tahun,"AWS, AWS Glue, Amazon Redshift, Amazon S3, Apache Beam, Apache Flink, Apache HTTP Server, Apache NiFi, Apache Spark, Big Data, Cloud Computing, Computer Science, Data Integration, Data Pipelines, ...","menguasai Python, Java, Scala, SQL, Hadoop, Apache HTTP Server Apache Spark, Amazon Amazon Redshift, Google Google BigQuery, Apache HTTP Server Beam, Apache HTTP Server Apache NiFi, AWS AWS Glue, ..."
1,Magister (S2),5-10 tahun,"AWS, Apache Beam, Apache Cassandra, Apache Flink, Apache HTTP Server, Apache Hive, Apache Pig, Apache Spark, BI, Big Data, Business Intelligence, Cloud Computing, Computer Science, D3.js, Data Int...","menguasai Python, Java, Scala, SQL, serta teknologi Big Data seperti Hadoop, Apache Spark, Apache Hive, Apache Pig; mampu merancang dan mengembangkan sistem data skala besar; bertanggung jawab dal..."
2,tidak ada pendidikan spesifik,10+ tahun,"Assembly, Business Analysis, Business Requirements, Change Management, ERP, FICO, Information Systems, Inventory Management, MM, PP, Relationship Management, SAP, SD, SRM, Supply Chain, System Int...","Menguasai Strategic Planning e Business-System Readiness, berpengalaman 10 tahun dalam bidang ERP, bertanggung jawab dalam Business Relationship Management, Training dan Change Management, Strateg..."
3,Sarjana (S1),10+ tahun,"Agile, Agile Methodology, Automation, BI, DBA, Disaster Recovery, Firewall, Query Analyzer, Query Optimization, Reporting, SQL, SQL Server, SSMS, SSRS, Statistics, Stored Procedures, T-SQL","menguasai SQL Server 2000-2017, berpengalaman 6 tahun di bidang database dengan fokus pada Performance Tuning, capacity planning, High Availability Solution, Disaster Recovery planning, serta meng..."
4,Magister (S2),5-10 tahun,"AWS, AWS Lambda, Agile, Agile Methodology, Amazon CloudWatch, Apache Cassandra, Azure Functions, C#, Cloud Computing, Cloud Cost Optimization, Cloud Functions, CloudTrail, Computer Science, Google...","Menguasai berbagai platform cloud, termasuk AWS, Microsoft Azure, dan Google Cloud Platform, dengan pengalaman 5+ tahun dalam Cloud Cost Optimization, Serverless Architecture, dan Cloud Computing,..."


---
# BAGIAN 6 — Pembersihan Manual (Setelah Ekspor)

Setelah seluruh pipeline otomatis selesai, file `Dataset_CV.xlsx` diperiksa secara manual
sebelum dianggap final.

Hal-hal yang perlu dicek:

| Kolom | Yang Perlu Diperiksa |
|---|---|
| `detail` | Kalimat tidak selesai, tidak relevan, atau masih janggal |
| `skill` | Baris dengan skill mirip tapi format berbeda (lolos dedup otomatis) |
| Semua | Baris duplikat tersembunyi (perbedaan spasi/koma kecil) |

> Tidak ada kode yang dijalankan di bagian ini.
> Proses ini dilakukan langsung di Excel.